### Sistema de Elasticidad Lineal

$$\begin{array}{rl}
-{\rm div}(\sigma({\bf u}))={\bf f} & \text{ en }\Omega \\
{\bf u} = {\bf 0} & \text{ en }\Gamma_4 \\
\sigma({\bf u})\cdot \vec{n} = {\bf 0} & \text{ en } \Gamma_1\cup \Gamma_2 \cup \Gamma_3
\end{array}
$$
con $\sigma({\bf u}) = 2\mu \epsilon({\bf u}) + \lambda {\rm tr}(\epsilon ({\bf u}))I$, y $\epsilon({\bf u}) = \frac12\left( \frac{\partial u_i}{\partial x_j} + \frac{\partial u_j}{\partial x_i} \right)$, y los parámetros de Lamé:
$$\lambda = \frac{\nu E}{(1+\nu)(1-2\nu)},\quad \mu = \frac{E}{2(1+\nu)},$$
siendo $E$ el módulo de Young, y $\nu$ el coeficiente de Poisson.

$\Omega = (10,0)\times (0,1)$, ${\bf f}=(0,-1)$, $E=21\times 10^5$, $\nu = 0.28$.

In [ ]:
%reset -f
import mfem.ser as mfem
from glvis import glvis

###  Malla inicial

In [ ]:
mesh = mfem.Mesh.MakeCartesian2D(100,10,mfem.Geometry.TRIANGLE,False, 10.,1.)

dim = mesh.Dimension()

print(f'Número de elementos: {mesh.GetNE():4d}')
print(f'Número de vértices: {mesh.GetNV():4d}')

glvis(mesh,keys='m******')

### Espacio de elementos finitos vectorial

In [ ]:
fec = mfem.H1_FECollection(2, dim)
fespace = mfem.FiniteElementSpace(mesh, fec, dim)

print(f'Número de incógnitas del espacio de elementos finitos: {fespace.GetTrueVSize():4d}')

### Condiciones frontera

In [ ]:
print("Etiquetas de la malla: " + str(mesh.bdr_attributes.ToList()))

Condición Dirichlet ${\bf u}={\bf 0}$ en la frontera 4

In [ ]:
ess_tdof_list = mfem.intArray()
border = [0]*mesh.bdr_attributes.Max()
# Condición Dirichlet en frontera 4
border[3] = 1
print(border)

Extraemos los nodos del espacio asociados a la condición frontera en `ess_tdof_list`

In [ ]:
ess_bdr = mfem.intArray(border)
fespace.GetEssentialTrueDofs(ess_bdr, ess_tdof_list)

### Fuerza aplicada
$$ {\bf f} = (0,-1) \text{ en }\Omega$$

In [ ]:
f = mfem.VectorArrayCoefficient(dim)
f.Set(0, mfem.ConstantCoefficient(0.0))
f.Set(1, mfem.ConstantCoefficient(-1.0))

Alternativamente:

In [ ]:
f = mfem.VectorConstantCoefficient(mfem.Vector([0.,-1.]))

### Formulación variacional 
$$\int_\Omega \left(\lambda {\rm div}({\bf u}) {\rm div}({\bf v}) + 2\mu \epsilon({\bf u}):\epsilon({\bf v})\right) = \int_\Omega {\bf f}\cdot {\bf v}$$

In [ ]:
E = 21.e5
nu = 0.28
mu = 1/(2*(1.+nu))
lamb = nu/ ( (1 + nu)*(1-2*nu) )

Ecoeff = mfem.ConstantCoefficient(E)

a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.ElasticityIntegrator(Ecoeff,lamb,mu))
a.Assemble()

b = mfem.LinearForm(fespace)
b.AddDomainIntegrator(mfem.VectorBoundaryLFIntegrator(f))
b.Assemble()


### Resolución
`u` es usada para pasar la condición Dirichlet, y luego para recuperar la solución como una `GridFunction`

In [ ]:
u = mfem.GridFunction(fespace)
u.Assign(0.0)

A = mfem.SparseMatrix()
B = mfem.Vector()
U = mfem.Vector()
a.FormLinearSystem(ess_tdof_list, u, b, A, U, B)
M = mfem.GSSmoother(A)

mfem.PCG(A, M, B, U, 0, 1500, 1e-8, 0.0)

a.RecoverFEMSolution(U,b,u)

### Visualización de resultados
Para visualizar el desplazamiento en la malla necesitamos mover los nodos según el deplazamiento. Para obtener los nodos vía una `GridFunction` usamos `SetNodalFESpace`, y a continuación podemos extraer los nodos con `GetNodes`

In [ ]:
deform_mesh = mfem.Mesh(mesh)
deform_mesh.SetNodalFESpace(fespace)
nodes = deform_mesh.GetNodes()

x = mfem.GridFunction(u) # deep copy
x *= 100
nodes += x

glvis((deform_mesh,u),keys="Rmjlcc***********")

### Extracción de componentes
Como la `GridFunction` representa un vector (las componentes están separadas en dos vectores por defecto, pero se puede hacer que en cada grado de libertad aparezcan consecutivamente los valores de cada componente: esto depende de que en la definición de espacio de elementos finitos especifiquemos el orden: byNODES (por defecto) o byVDIM).

Si lo que queremos es obtener una `GridFunction` con toda la componente debemos recuperar los datos del vector mediante `GetDataArray`

In [ ]:
fesp = mfem.FiniteElementSpace(mesh, fec) # fesp = H2
u1 = mfem.GridFunction(fesp,mfem.Vector(u.GetDataArray()))
u2 = mfem.GridFunction(fesp,mfem.Vector(u.GetDataArray()),fesp.GetVSize())

Alternativa:

In [ ]:
u1 = mfem.GridFunction()
u2 = mfem.GridFunction()
u1.MakeRef(fesp, u, 0)
u2.MakeRef(fesp, u ,fesp.GetVSize())

##### Valores en los nodos

Podemos extraer sólo el valor en los nodos de la malla (con independencia del orden de aproximación) con `GetNodalValues`. `y1`, `y2` son vectores que recuperan los valores en los nodos de la solución `u`

In [ ]:
y1 = mfem.Vector()
y2 = mfem.Vector()
u.GetNodalValues(y1,1)
u.GetNodalValues(y2,2)

### Interpolación en otros espacios
Ahora podemos construir una función en P1 (nótese que `u` está en P2), con los valores obtenidos en `y1`

In [ ]:
fec0 = mfem.H1_FECollection(1, dim)
fespa = mfem.FiniteElementSpace(mesh, fec0) # fespa = H1

z1 = mfem.GridFunction(fespa,y1)

También es posible proyectar una función de un espacio en otro, con `ProjectGridFunction`

In [ ]:
w1 = mfem.GridFunction(fespa)
w1.ProjectGridFunction(u1) # proyecta los valores de u1 en w1

Comparamos la diferencia entre ambas funciones $\Vert w_1 - z_1\Vert_\infty$, $\Vert w_1 - z_1\Vert_2$, 

In [ ]:
zz1 = mfem.GridFunctionCoefficient(z1)
print(w1.ComputeL2Error(zz1))
print(w1.ComputeMaxError(zz1))

También lo podemos hacer projectando desde P1 a P2:

In [ ]:
z2 = mfem.GridFunction(fesp) # función en P2

In [ ]:
w2 = mfem.GridFunction(fespa,y2) # define un función en P1 con los valores de y2
z2.ProjectGridFunction(w2) # proyecta los valores de w2 en z2 (de P1 a P2)

Comparamos la proyección en P2 con la original

In [ ]:
zz2 = mfem.GridFunctionCoefficient(z2)
print(u2.ComputeMaxError(zz2))
print(u2.ComputeL2Error(zz2))